# 🚀 Laya-Multilingual: Zero-Shot Baseline Evaluation & Fine-Tuning on 9,000 Fatwa Decisions
### Measuring Before vs. After on 1,350 Held-Out Decisions + Automated Hugging Face Push

This notebook runs the complete end-to-end evaluation & training pipeline on the **9,000 Fatwa Relevance dataset**:
1. **Stratified Split**: 85% Train (7,650 items) / 15% Held-Out Test (1,350 items).
2. **Phase 1 — Zero-Shot Baseline Benchmark**: Evaluates the untuned base model (`convaiinnovations/laya-multilingual`) on the 1,350 held-out test items.
3. **Phase 2 — Multi-GPU Training (`train_ddp.py`)**: Runs DDP RLCD training with temperature calibration on 7,650 items.
4. **Phase 3 — Post-Training Evaluation**: Evaluates the fine-tuned model on the exact same 1,350 test items.
5. **Phase 4 — Head-to-Head Comparison Table**: Directly reports the before-and-after accuracy lift, recall improvement, and latency.
6. **Phase 5 — Automated Hugging Face Hub Upload**: Pushes model weights, config, tokenizer, and generated Model Card to Hugging Face Hub.

---

### Hardware Requirements
- **Accelerator**: GPU T4 × 2 (recommended for fast DDP training) or GPU T4/P100 × 1.
- **Internet**: **ON** (Right sidebar -> Settings -> Internet -> On).


In [ ]:
# 1. Environment & GPU Verification
!nvidia-smi
import torch
print(f"\nPyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
print(f"GPU Count:       {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  • GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / (1024**3):.1f} GB)")


In [ ]:
# 2. Install Laya & Dependencies
!pip install -q "laya>=0.1.6" "transformers>=4.48.0" "datasets>=3.0.0" safetensors huggingface_hub accelerate tabulate scikit-learn

import laya
import transformers
print("✓ Laya version        :", laya.__version__)
print("✓ Transformers version:", transformers.__version__)
print("✓ PyTorch version     :", torch.__version__)


In [ ]:
# 3. Load & Preprocess 9,000 Dataset for Training & Evaluation
import os
import sys
import json
import zipfile
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer
from huggingface_hub import snapshot_download
from laya.agent import _fix_tokenizer_config
from laya.train import items_from_rows

MODEL_ID = "convaiinnovations/laya-multilingual"
print(f"1. Downloading base model snapshot & tokenizer from Hugging Face ({MODEL_ID})...")
model_dir = snapshot_download(MODEL_ID)
_fix_tokenizer_config(model_dir)

tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
with open(os.path.join(model_dir, "rl_agent_config.json"), "r", encoding="utf-8") as f:
    cfg = json.load(f)

# 2. Locate 9k dataset
DATA_SOURCE = None
search_paths = [
    Path("/kaggle/working/dataset_9k_bundle.zip"),
    Path("/kaggle/working/dataset_9k_labeled.json"),
    Path("/kaggle/working/dataset_9k_labeled.csv"),
    Path("outputs/dataset_9k_bundle.zip"),
    Path("outputs/dataset_9k_labeled.json"),
]
for p in Path("/kaggle/input").rglob("*"):
    if p.name in ("dataset_9k_bundle.zip", "dataset_9k_labeled.json", "dataset_9k_labeled.csv"):
        search_paths.insert(0, p)

for sp in search_paths:
    if sp.exists():
        DATA_SOURCE = sp
        break

assert DATA_SOURCE is not None, "Could not find dataset_9k_bundle.zip or dataset_9k_labeled.json! Please attach it under Input."
print(f"✓ Found dataset source at: {DATA_SOURCE}")

raw_items = []
if DATA_SOURCE.name.endswith(".zip"):
    with zipfile.ZipFile(DATA_SOURCE, "r") as zf:
        if "dataset_9k_labeled.json" in zf.namelist():
            with zf.open("dataset_9k_labeled.json") as f:
                raw_items = json.load(f)
        elif "dataset_9k_labeled.csv" in zf.namelist():
            with zf.open("dataset_9k_labeled.csv") as f:
                raw_items = pd.read_csv(f).to_dict(orient="records")
elif DATA_SOURCE.name.endswith(".json"):
    with open(DATA_SOURCE, "r", encoding="utf-8") as f:
        raw_items = json.load(f)
elif DATA_SOURCE.name.endswith(".csv"):
    raw_items = pd.read_csv(DATA_SOURCE).to_dict(orient="records")

if isinstance(raw_items, dict):
    raw_items = list(raw_items.values())

print(f"✓ Loaded {len(raw_items):,} labeled items.")

# 3. Canonical format: {state, questions, gold, expected}
QUESTION_SCHEMA = {
    "relevance": {
        "type": "choice",
        "instructions": (
            "Determine whether the Islamic fatwa directly and specifically answers the user's question.\n"
            "- 'answers-it': The fatwa directly and specifically answers the question asked.\n"
            "- 'does-not-answer': The fatwa does NOT directly answer the question (touches on a related topic or is unrelated)."
        ),
        "criteria": {
            "answers-it": "The fatwa directly answers the specific question asked.",
            "does-not-answer": "The fatwa does not directly answer the question.",
        }
    }
}

canonical_rows = []
for it in raw_items:
    v = str(it.get("gpt_verdict", it.get("verdict", ""))).strip().lower().replace("_", "-")
    if v not in ("answers-it", "does-not-answer"):
        continue
    conf = float(it.get("gpt_confidence", it.get("confidence", 1.0)))
    conf = min(max(conf, 0.51), 1.0)
    other_conf = round(1.0 - conf, 4)
    
    state_dict = {
        "user_query": str(it.get("question", "")).strip(),
        "scholar": str(it.get("scholar", "")).strip(),
        "fatwa_title": str(it.get("title", "")).strip(),
        "fatwa_answer": str(it.get("answer_text", ""))[:1400].strip(),
    }
    
    canonical_rows.append({
        "state": state_dict,
        "questions": QUESTION_SCHEMA,
        "gold": {
            "relevance": {
                "label": v,
                "probabilities": {
                    "answers-it": conf if v == "answers-it" else other_conf,
                    "does-not-answer": conf if v == "does-not-answer" else other_conf,
                }
            }
        },
        "expected": {
            "relevance": v
        },
        "metadata": {
            "id": it.get("id"),
            "scholar": str(it.get("scholar", "")).strip(),
            "category": str(it.get("category", "General")).strip(),
            "verdict": v,
        }
    })

print(f"✓ Valid canonical examples: {len(canonical_rows):,}")

# 4. Stratified 85% Train / 15% Held-Out Test Split
labels = [r["expected"]["relevance"] for r in canonical_rows]
train_rows, test_rows = train_test_split(
    canonical_rows,
    test_size=0.15,
    random_state=42,
    stratify=labels
)

print(f"• Training Pool: {len(train_rows):,} rows ({sum(1 for r in train_rows if r['expected']['relevance'] == 'answers-it') / len(train_rows)*100:.1f}% positives)")
print(f"• Test Pool:     {len(test_rows):,} rows ({sum(1 for r in test_rows if r['expected']['relevance'] == 'answers-it') / len(test_rows)*100:.1f}% positives)")

with open("/kaggle/working/test_rows.json", "w", encoding="utf-8") as f:
    json.dump(test_rows, f, ensure_ascii=False, indent=2)

# 5. Tokenize training items
cfg["max_len"] = 1024
cfg["head_max_len"] = 256
print(f"\nTokenizing training items (max_len={cfg['max_len']}, head_max_len={cfg['head_max_len']})...")
train_items, skipped = items_from_rows(tok, train_rows, cfg["max_len"], cfg["head_max_len"])
print(f"✓ Preprocessed {len(train_items):,} training items (skipped: {skipped}).")

torch.save(train_items, "/kaggle/working/train_items.pt")
print("✓ Saved preprocessed training items to /kaggle/working/train_items.pt")


In [ ]:
# 4. Phase 1: Zero-Shot Baseline Benchmark on Held-Out Test Split (Untuned Laya)
import time
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, confusion_matrix

print("Loading raw zero-shot base model on GPU...")
device = "cuda" if torch.cuda.is_available() else "cpu"
base_agent = laya.load(MODEL_ID, device=device)

with open("/kaggle/working/test_rows.json", "r", encoding="utf-8") as f:
    test_rows = json.load(f)

print(f"Evaluating {len(test_rows):,} test cases with Base Zero-Shot Laya...")
base_preds = []
base_latencies = []

t0_base = time.time()
for item in tqdm(test_rows, desc="Base Model Evaluation"):
    t0 = time.perf_counter()
    res = base_agent.predict(item["state"], item["questions"])
    dt_ms = (time.perf_counter() - t0) * 1000
    base_latencies.append(dt_ms)
    
    ans = res["answers"].get("relevance", {})
    pred = ans.get("choice", "unknown")
    gold = item["expected"]["relevance"]
    
    base_preds.append({
        "scholar": item["metadata"]["scholar"],
        "gold": gold,
        "pred": pred,
        "confidence": ans.get("confidence", 0.0),
        "latency_ms": dt_ms,
        "correct": int(pred == gold),
    })

base_df = pd.DataFrame(base_preds)
base_acc = base_df["correct"].mean()
base_p50 = np.percentile(base_latencies, 50)
print(f"\n✓ Base Zero-Shot Accuracy on {len(base_df):,} held-out items: {base_acc * 100:.2f}% (Median Latency: {base_p50:.2f} ms)")


In [ ]:
%%writefile /kaggle/working/train_ddp.py
"""DDP & Single-GPU Training script for Laya-Multilingual fine-tuning on 9,000 Fatwa Dataset.
"""
import os
import sys
import time
import json
import random
import math
import numpy as np
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from safetensors.torch import load_file
from transformers import AutoTokenizer

from laya.common import build_model, collate_items
from laya.calibrate import fit_temperature_map
from laya.train import (
    calibration_records,
    encode_item,
    rlcd_loss,
    save_checkpoint,
    sigma_at,
    split_calibration,
)

def main():
    dist.init_process_group("nccl")
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)

    model_dir = sys.argv[1]
    output_dir = sys.argv[2]
    os.makedirs(output_dir, exist_ok=True)

    with open(os.path.join(model_dir, "rl_agent_config.json"), "r", encoding="utf-8") as f:
        cfg = json.load(f)
    cfg["gradient_checkpointing"] = True
    cfg["max_tokens_per_batch"] = 4096
    cfg["max_len"] = 1024
    cfg["head_max_len"] = 256
    max_len = cfg["max_len"]
    head_max_len = cfg["head_max_len"]

    tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
    model = build_model(cfg, encoder_dir=os.path.join(model_dir, "encoder"))

    weights = load_file(os.path.join(model_dir, "model.safetensors"))
    model.load_state_dict(weights, strict=True)

    model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.head_checkpointing = True
    model.to(device)
    model.train()

    if world_size > 1:
        ddp_model = DDP(model, device_ids=[local_rank], find_unused_parameters=True)
    else:
        ddp_model = model

    all_items = torch.load("/kaggle/working/train_items.pt", weights_only=False)

    # Calibration holdout: seed-deterministic and rank-independent
    train_items, calib_items = split_calibration(all_items, calib_max=400, calib_frac=0.08, seed=20261027)

    # Trim to multiple of world_size
    train_items = train_items[:len(train_items) // world_size * world_size]
    my_items = train_items[rank::world_size]

    EPOCHS = 4
    MICRO_BATCH = 8
    GRAD_ACCUM = 4
    GROUP_SIZE = 4
    LR_ENCODER = 2.0e-5
    LR_HEAD = 1.0e-4
    SIGMA_START = 0.4
    SIGMA_END = 0.1

    enc_params = [p for n, p in ddp_model.named_parameters() if "encoder." in n]
    head_params = [p for n, p in ddp_model.named_parameters() if "encoder." not in n]

    optimizer = torch.optim.AdamW([
        {"params": enc_params, "lr": LR_ENCODER},
        {"params": head_params, "lr": LR_HEAD}
    ], weight_decay=0.01)

    total_updates = (len(my_items) // (MICRO_BATCH * GRAD_ACCUM)) * EPOCHS
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, total_updates), eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda", enabled=True)

    if rank == 0:
        print(f"Starting Training: {len(train_items)} train items ({len(calib_items)} calib holdout) | {world_size} GPU(s) | {EPOCHS} epochs")
    t0 = time.time()

    for epoch in range(EPOCHS):
        random.seed(42 + epoch + rank)
        random.shuffle(my_items)
        epoch_loss, n_batches = 0.0, 0
        optimizer.zero_grad(set_to_none=True)
        accum_step = 0

        sigma = sigma_at(epoch, EPOCHS, SIGMA_START, SIGMA_END)
        steps_per_epoch = math.ceil(len(my_items) / MICRO_BATCH)

        for b_idx in range(0, len(my_items), MICRO_BATCH):
            chunk = my_items[b_idx:b_idx + MICRO_BATCH]
            if not chunk:
                continue

            encoded = [encode_item(tok, it, max_len, head_max_len) for it in chunk]
            batch = collate_items([encoded], tok.pad_token_id)

            with torch.autocast("cuda", dtype=torch.float16):
                logits, act = ddp_model(
                    batch["input_ids"].to(device),
                    batch["attention_mask"].to(device),
                    batch["marker_pos"].to(device),
                    batch["marker_mask"].to(device),
                    batch["qtype"].to(device)
                )

            logits = logits.float()
            mask = batch["marker_mask"].to(device)
            target = batch["target"].to(device)

            loss = rlcd_loss(logits, target, mask, batch["qtype"].to(device), sigma,
                             samples=GROUP_SIZE, w_sph=0.75, w_rps=1.0)

            window_start = (accum_step // GRAD_ACCUM) * GRAD_ACCUM
            window_size = min(GRAD_ACCUM, steps_per_epoch - window_start)
            scaled = loss / window_size
            scaler.scale(scaled).backward()
            accum_step += 1

            if accum_step % GRAD_ACCUM == 0 or (b_idx + MICRO_BATCH) >= len(my_items):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(ddp_model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            epoch_loss += loss.item()
            n_batches += 1

            if rank == 0 and (n_batches % 50) == 0:
                cur_lr = scheduler.get_last_lr()[0]
                print(f"  Epoch {epoch+1}/{EPOCHS} | Step {n_batches} | Loss: {loss.item():.4f} | LR: {cur_lr:.2e}")

        if rank == 0:
            print(f"=== Epoch {epoch+1}/{EPOCHS} Completed in {time.time()-t0:.1f}s | Avg Loss: {epoch_loss/max(1, n_batches):.4f} ===")

        dist.barrier()

    # Temperature Calibration on Rank 0
    if rank == 0:
        print("\nFitting post-training temperature calibration...")
        del optimizer, scaler, scheduler
        torch.cuda.empty_cache()
        model.eval()
        try:
            records = calibration_records(model, tok, calib_items, device, max_len, head_max_len, batch_size=16)
            fitted = fit_temperature_map(records)
            fitted_temps = fitted["temperature"]
            print("Fitted calibration temperatures (choice, score, noul):", [round(t, 3) for t in fitted_temps])
        except Exception as e:
            print("Temperature fitting fallback:", e)
            fitted_temps, fitted = None, {"temperature_by_options": {}}

        out_cfg = dict(cfg, max_len=max_len, head_max_len=head_max_len)
        out_cfg["fine_tuned"] = True
        out_cfg["model_name"] = "laya-multilingual-fatwa-relevance-9k"
        if fitted_temps is not None:
            out_cfg["temperature"] = fitted_temps
            out_cfg.pop("temperature_by_options", None)
            if fitted["temperature_by_options"]:
                out_cfg["temperature_by_options"] = fitted["temperature_by_options"]

        save_checkpoint(model, tok, out_cfg, output_dir)
        print(f"✓ Model successfully saved to {output_dir}!")

    dist.destroy_process_group()

if __name__ == "__main__":
    main()


In [ ]:
# 6. Launch Multi-GPU / Single-GPU Training with torchrun
OUTPUT_DIR = "/kaggle/working/laya_finetuned_fatwa_relevance_9k"
MODEL_DIR = model_dir

n_gpus = max(1, torch.cuda.device_count())
cmd = f"torchrun --standalone --nproc_per_node={n_gpus} /kaggle/working/train_ddp.py {MODEL_DIR} {OUTPUT_DIR}"
print(f"Executing Training on {n_gpus} GPU(s):\n{cmd}\n")

!{cmd}


In [ ]:
# 7. Phase 3: Evaluate Fine-Tuned Model on the SAME Held-Out Test Set
OUTPUT_DIR = "/kaggle/working/laya_finetuned_fatwa_relevance_9k"
print("Loading fine-tuned Laya Agent on GPU...")
agent_ft = laya.Agent(OUTPUT_DIR, device="cuda")

print(f"Evaluating {len(test_rows):,} test cases with fine-tuned model...")
ft_preds = []
ft_latencies = []

t0_ft = time.time()
for item in tqdm(test_rows, desc="Fine-Tuned Model Evaluation"):
    t0 = time.perf_counter()
    res = agent_ft.predict(item["state"], item["questions"])
    dt_ms = (time.perf_counter() - t0) * 1000
    ft_latencies.append(dt_ms)
    
    ans = res["answers"].get("relevance", {})
    pred = ans.get("choice", "unknown")
    gold = item["expected"]["relevance"]
    
    ft_preds.append({
        "scholar": item["metadata"]["scholar"],
        "gold": gold,
        "pred": pred,
        "confidence": ans.get("confidence", 0.0),
        "latency_ms": dt_ms,
        "correct": int(pred == gold),
    })

ft_df = pd.DataFrame(ft_preds)
ft_acc = ft_df["correct"].mean()
ft_p50 = np.percentile(ft_latencies, 50)
print(f"\n✓ Fine-Tuned Accuracy on {len(ft_df):,} held-out items: {ft_acc * 100:.2f}% (Median Latency: {ft_p50:.2f} ms)")


In [ ]:
# 8. Head-to-Head Comparison: Before (Zero-Shot Base) vs. After (Fine-Tuned 9K)
from sklearn.metrics import classification_report

print("=" * 80)
print("📊 HEAD-TO-HEAD BENCHMARK: ZERO-SHOT BASE vs. FINE-TUNED 9K")
print("=" * 80)

# Metrics comparison table
comp_data = [
    {
        "Metric": "Overall Accuracy",
        "Base Laya (Zero-Shot)": f"{base_acc * 100:.2f}%",
        "Fine-Tuned Laya (9K)": f"{ft_acc * 100:.2f}%",
        "Absolute Gain": f"+{(ft_acc - base_acc) * 100:.2f}%"
    },
    {
        "Metric": "Median Latency (p50)",
        "Base Laya (Zero-Shot)": f"{base_p50:.2f} ms",
        "Fine-Tuned Laya (9K)": f"{ft_p50:.2f} ms",
        "Absolute Gain": f"{ft_p50 - base_p50:+.2f} ms"
    },
]
display(pd.DataFrame(comp_data))

print("\n" + "=" * 80)
print("CLASSIFICATION REPORT: BASE LAYA (ZERO-SHOT):")
print("=" * 80)
print(classification_report(base_df["gold"], base_df["pred"], digits=3))

print("\n" + "=" * 80)
print("CLASSIFICATION REPORT: FINE-TUNED LAYA (9K):")
print("=" * 80)
print(classification_report(ft_df["gold"], ft_df["pred"], digits=3))

print("\n" + "=" * 80)
print("SCHOLAR ACCURACY COMPARISON:")
print("=" * 80)
s_base = base_df.groupby("scholar")["correct"].agg(["count", "mean"]).rename(columns={"mean": "Base Accuracy"})
s_ft = ft_df.groupby("scholar")["correct"].agg(["mean"]).rename(columns={"mean": "FineTuned Accuracy"})
s_comp = s_base.join(s_ft)
s_comp["Base Accuracy"] = s_comp["Base Accuracy"].map(lambda x: f"{x*100:.1f}%")
s_comp["FineTuned Accuracy"] = s_comp["FineTuned Accuracy"].map(lambda x: f"{x*100:.1f}%")
display(s_comp.reset_index())


In [ ]:
# 9. Upload 9K Fine-Tuned Model to Hugging Face Hub
import os
import getpass
import zipfile
from huggingface_hub import HfApi

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    if HF_TOKEN:
        print("✓ Loaded HF_TOKEN from Kaggle Secrets.")
except Exception:
    pass

if not HF_TOKEN:
    HF_TOKEN = os.getenv("HF_TOKEN")

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("🔑 Enter your Hugging Face write token: ").strip()

assert HF_TOKEN and HF_TOKEN.startswith("hf_"), "Valid Hugging Face token required!"

api = HfApi(token=HF_TOKEN)
user_info = api.whoami()
hf_username = user_info["name"]
repo_name = f"{hf_username}/laya-multilingual-fatwa-relevance"

print(f"✓ Authenticated as: {hf_username}")
print(f"Target repository: {repo_name}")

readme_content = f"""---
license: apache-2.0
library_name: transformers
tags:
- laya
- system-one
- islamic-fatwa
- relevance-classification
- rag
- arabic
- salafi-scholars
- rlcd
metrics:
- accuracy
model-index:
- name: laya-multilingual-fatwa-relevance
  results:
  - task:
      type: text-classification
      name: Islamic Fatwa Relevance Classification (9K Benchmark)
    metrics:
    - type: accuracy
      value: {ft_acc:.3f}
---

# Laya Multilingual: Fine-Tuned on 9,000 Islamic Fatwa Decisions

This model is fine-tuned from **`convaiinnovations/laya-multilingual`** (322M mmBERT non-autoregressive decision engine) on a curated, balanced **9,000 Islamic jurisprudence dataset** across 7 major Salafi scholars:
- **Sheikh Abdulaziz Ibn Baz** (الشيخ عبد العزيز بن باز)
- **Sheikh Muhammad ibn al-Uthaymeen** (الشيخ محمد بن صالح العثيمين)
- **Al-Lajnah Ad-Da'imah** (اللجنة الدائمة للبحوث العلمية والإفتاء)
- **Sheikh Saleh Al-Fawzan** (الشيخ صالح الفوزان)
- **Sheikh Abdullah al-Manee'** (الشيخ عبد الله بن منيع)
- **Sheikh Abdulaziz Aal ash-Sheikh** (الشيخ عبد العزيز آل الشيخ)
- **Sheikh Saleh bin Humaid** (الشيخ صالح بن حميد)

## Performance Lift
- **Zero-Shot Baseline Accuracy**: {base_acc*100:.2f}%
- **Fine-Tuned 9K Accuracy**: **{ft_acc*100:.2f}%** (**+{(ft_acc-base_acc)*100:.2f}% lift**)
- **Inference Latency**: **~{ft_p50:.1f} ms** per decision on GPU (single forward pass, non-autoregressive)
- **Cost**: **$0.00** (completely self-hosted, 0 API tokens required)

## Usage Quickstart

```bash
pip install laya
```

```python
import laya

# Load directly from Hugging Face
agent = laya.load("{repo_name}")

state = {{
    "user_query": "هل يجوز الجمع بين الصلاتين بسبب المطر الشديد؟",
    "scholar": "Ibn Baz",
    "fatwa_title": "الجمع بين المغرب والعشاء للمطر",
    "fatwa_answer": "يجوز الجمع بين المغرب والعشاء للمطر الشديد الذي يشق على الناس..."
}}

questions = {{
    "relevance": {{
        "type": "choice",
        "instructions": "Determine whether the fatwa answers the user question directly.",
        "criteria": {{
            "answers-it": "The fatwa directly answers the specific question asked.",
            "does-not-answer": "The fatwa does not directly answer the question."
        }}
    }}
}}

result = agent.predict(state, questions)
ans = result["answers"]["relevance"]
print("Verdict:   ", ans["choice"])        # 'answers-it' or 'does-not-answer'
print("Confidence:", ans["confidence"])    # e.g. 0.98
print("Probs:     ", ans["probabilities"])
```
"""

with open(os.path.join(OUTPUT_DIR, "README.md"), "w", encoding="utf-8") as f:
    f.write(readme_content)

print(f"Uploading fine-tuned model artifacts to https://huggingface.co/{repo_name}...")
api.create_repo(repo_name, repo_type="model", exist_ok=True)
api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id=repo_name,
    repo_type="model",
    commit_message=f"Update fine-tuned Laya Multilingual on 9k dataset (Accuracy: {ft_acc*100:.1f}%)"
)
print(f"\n🎉 MODEL PUBLISHED SUCCESSFULLY! Check it out at:\n👉 https://huggingface.co/{repo_name}")

# Create local ZIP archive in /kaggle/working/
zip_path = "/kaggle/working/laya_finetuned_fatwa_relevance_9k.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(OUTPUT_DIR):
        for f in files:
            full_p = os.path.join(root, f)
            arc_p = os.path.relpath(full_p, OUTPUT_DIR)
            zf.write(full_p, arc_p)

print(f"✓ Backup ZIP archive saved to: {zip_path} ({os.path.getsize(zip_path)/(1024**2):.1f} MB)")
